## Ontology Builder

In [8]:
!pip install rdflib

In [ ]:
from rdflib import Graph, Literal, Namespace, RDF, RDFS
from rdflib.namespace import OWL, XSD
import re

BMO = Namespace("http://www.bridge-management-ontology.org/bmo#")
g = Graph()
g.bind("bmo", BMO)
g.bind("owl", OWL)
g.bind("rdfs", RDFS)
g.bind("xsd", XSD)

top_level_classes = {
    "Asset": "A physical entity that is owned and has value, such as a bridge or its components.",
    "Event": "An occurrence that happens at a specific point in time related to an asset.",
    "ExternalFactor": "An external condition or influence that affects an asset."
}
for name, comment in top_level_classes.items():
    class_uri = BMO[name]
    g.add((class_uri, RDF.type, OWL.Class))
    g.add((class_uri, RDFS.label, Literal(name)))
    g.add((class_uri, RDFS.comment, Literal(comment)))


bridge_class = BMO.Bridge
g.add((bridge_class, RDF.type, OWL.Class))
g.add((bridge_class, RDFS.subClassOf, BMO.Asset))
g.add((bridge_class, RDFS.label, Literal("Bridge")))

bridge_element_class = BMO.BridgeElement
g.add((bridge_element_class, RDF.type, OWL.Class))
g.add((bridge_element_class, RDFS.subClassOf, BMO.Asset))
g.add((bridge_element_class, RDFS.label, Literal("Bridge Element")))

inspection_event_class = BMO.InspectionEvent
g.add((inspection_event_class, RDF.type, OWL.Class))
g.add((inspection_event_class, RDFS.subClassOf, BMO.Event))
g.add((inspection_event_class, RDFS.label, Literal("Inspection Event")))

maintenance_event_class = BMO.MaintenanceEvent
g.add((maintenance_event_class, RDF.type, OWL.Class))
g.add((maintenance_event_class, RDFS.subClassOf, BMO.Event))
g.add((maintenance_event_class, RDFS.label, Literal("Maintenance Event")))

traffic_profile_class = BMO.TrafficProfile
g.add((traffic_profile_class, RDF.type, OWL.Class))
g.add((traffic_profile_class, RDFS.subClassOf, BMO.ExternalFactor))
g.add((traffic_profile_class, RDFS.label, Literal("Traffic Profile")))

environmental_exposure_class = BMO.EnvironmentalExposure
g.add((environmental_exposure_class, RDF.type, OWL.Class))
g.add((environmental_exposure_class, RDFS.subClassOf, BMO.ExternalFactor))
g.add((environmental_exposure_class, RDFS.label, Literal("Environmental Exposure")))



# A Bridge has Bridge Elements
g.add((BMO.hasElement, RDF.type, OWL.ObjectProperty))
g.add((BMO.hasElement, RDFS.domain, bridge_class))
g.add((BMO.hasElement, RDFS.range, bridge_element_class))

# An Asset undergoes an Event
g.add((BMO.undergoesEvent, RDF.type, OWL.ObjectProperty))
g.add((BMO.undergoesEvent, RDFS.domain, BMO.Asset))
g.add((BMO.undergoesEvent, RDFS.range, BMO.Event))

# An Asset is influenced by an External Factor
g.add((BMO.isInfluencedBy, RDF.type, OWL.ObjectProperty))
g.add((BMO.isInfluencedBy, RDFS.domain, BMO.Asset))
g.add((BMO.isInfluencedBy, RDFS.range, BMO.ExternalFactor))



def create_data_property(column_name, domain_class, data_type=XSD.string):
    """Helper function to create a data property and link it to its domain class."""
    prop_name_camel = ''.join(x.capitalize() for x in re.split(r'[\s\(\).\-/%;?]', column_name) if x)
    prop_name_camel = prop_name_camel[0].lower() + prop_name_camel[1:] if prop_name_camel else column_name.lower()
    
    prop_uri = BMO[prop_name_camel]
    g.add((prop_uri, RDF.type, OWL.DatatypeProperty))
    g.add((prop_uri, RDFS.label, Literal(column_name)))
    g.add((prop_uri, RDFS.domain, domain_class))
    g.add((prop_uri, RDFS.range, data_type))

# --- Asset -> Bridge Properties (Structural, Location, Admin) ---
create_data_property("STRUCTURE_NUMBER", bridge_class, XSD.string)
create_data_property("NBI_27 - Year Built", bridge_class, XSD.gYear)
create_data_property("NBI_106 - Year Reconstructed", bridge_class, XSD.gYear)
create_data_property("NBI_Bridge Age (yr)", bridge_class, XSD.integer)

# Structural Properties
create_data_property("NBI_43A - Main Span Material", bridge_class, XSD.string)
create_data_property("NBI_44A - Approach Spans Material", bridge_class, XSD.string)
create_data_property("NBI_46 - Number of Approach Spans", bridge_class, XSD.integer)
create_data_property("NBI_48 - Length of Maximum Span (ft.)", bridge_class, XSD.decimal)
create_data_property("NBI_49 - Structure Length (ft.)", bridge_class, XSD.decimal)
create_data_property("NBI_112 - NBIS Bridge Length", bridge_class, XSD.string) # NBIS length can be a flag
create_data_property("NBI_51 - Bridge Roadway Width - Curb to Curb (ft.)", bridge_class, XSD.decimal)
create_data_property("NBI_52 - Deck Width - Out to Out (ft.)", bridge_class, XSD.decimal)
create_data_property("NBI_CAT29 - Deck Area (sq. ft.)", bridge_class, XSD.decimal)
create_data_property("NBI_107 - Deck Structure Type Code", bridge_class, XSD.string)
create_data_property("NBI_108A - Wearing Surface Type Code", bridge_class, XSD.string)
create_data_property("NBI_108B - Membrane Type Code", bridge_class, XSD.string)
create_data_property("NBI_31 - Design Load", bridge_class, XSD.string)
create_data_property("NBI_32 - Approach Roadway Width (ft.)", bridge_class, XSD.decimal)
create_data_property("NBI_10 - Inventory Route, Minimum Vertical Clearance (ft.)", bridge_class, XSD.decimal)

# Location & Admin Properties
create_data_property("OWNER_AGENCY", bridge_class, XSD.string)
create_data_property("NBI_1 - State Code", bridge_class, XSD.string)
create_data_property("NBI_3 - County Code", bridge_class, XSD.string)
create_data_property("NBI_4 - Place Code", bridge_class, XSD.string)
create_data_property("NBI_9 - Administrative Area", bridge_class, XSD.string)
create_data_property("NBI_21 - Maintenance Responsibility", bridge_class, XSD.string)
create_data_property("NBI_26 - Functional Classification of Inventory Route", bridge_class, XSD.string)
create_data_property("NBI_104 - Highway System of Inventory Route", bridge_class, XSD.string)
create_data_property("NBI_37 - Historical Significance Code", bridge_class, XSD.string)


# --- Asset -> BridgeElement Properties ---
create_data_property("ELEMENT_CATEGORY", bridge_element_class, XSD.string)
create_data_property("PARENT_ELEMENT_NUMBER", bridge_element_class, XSD.string)
create_data_property("ELEMENT_NAME", bridge_element_class, XSD.string)
create_data_property("ELEMENT_UNIT", bridge_element_class, XSD.string)
create_data_property("TOTAL_QUANTITY", bridge_element_class, XSD.decimal)
create_data_property("CONDITION_STATE_1", bridge_element_class, XSD.decimal) 
create_data_property("CS_PERCENT_1", bridge_element_class, XSD.decimal)
create_data_property("CONDITION_STATE_2", bridge_element_class, XSD.decimal)
create_data_property("CS_PERCENT_2", bridge_element_class, XSD.decimal)
create_data_property("CONDITION_STATE_3", bridge_element_class, XSD.decimal)
create_data_property("CS_PERCENT_3", bridge_element_class, XSD.decimal)
create_data_property("CONDITION_STATE_4", bridge_element_class, XSD.decimal)
create_data_property("CS_PERCENT_4", bridge_element_class, XSD.decimal)
create_data_property("HI", bridge_element_class, XSD.decimal) 

# --- Event -> InspectionEvent Properties ---
create_data_property("NBI_90 - Inspection Date", inspection_event_class, XSD.date)
create_data_property("NBI_91 - Designated Inspection Frequency", inspection_event_class, XSD.integer)
create_data_property("NBI_CAT10 - Bridge Condition", inspection_event_class, XSD.string)
create_data_property("NBI_92A - Fracture Critical Details", inspection_event_class, XSD.string)
create_data_property("NBI_92B - Underwater Inspection", inspection_event_class, XSD.string)
create_data_property("NBI_92C - Other Special Inspection", inspection_event_class, XSD.string)

# --- Event -> MaintenanceEvent Properties ---
create_data_property("NBI_94 - Bridge Improvement Cost", maintenance_event_class, XSD.decimal)
create_data_property("NBI_95 - Roadway Improvement Cost", maintenance_event_class, XSD.decimal)
create_data_property("NBI_76 - Length of Structure Improvement (ft.)", maintenance_event_class, XSD.decimal)

# --- ExternalFactor -> TrafficProfile Properties ---
create_data_property("NBI_29 - Average Daily Traffic", traffic_profile_class, XSD.integer)
create_data_property("NBI_109 - Average Daily Truck Traffic (%)", traffic_profile_class, XSD.decimal) # Changed to decimal
create_data_property("NBI_30 - Year of Average Daily Traffic", traffic_profile_class, XSD.gYear)

# --- ExternalFactor -> EnvironmentalExposure Properties ---
create_data_property("NBI_Avg Daily Max Temp", environmental_exposure_class, XSD.decimal)
create_data_property("NBI_Avg Daily Min Temp", environmental_exposure_class, XSD.decimal)
create_data_property("NBI_Avg Daily Wind Speed", environmental_exposure_class, XSD.decimal)
create_data_property("NBI_Avg Monthly Max Temp", environmental_exposure_class, XSD.decimal)
create_data_property("NBI_Avg Monthly Min Temp", environmental_exposure_class, XSD.decimal)
create_data_property("NBI_Freeze Thaw Cycles", environmental_exposure_class, XSD.integer)
create_data_property("NBI_Number of Days with Measurable Precipitation", environmental_exposure_class, XSD.integer)
create_data_property("NBI_Number of Days with Temperature Below 0?C", environmental_exposure_class, XSD.integer)
create_data_property("NBI_Number of Snowfall Days", environmental_exposure_class, XSD.integer)
create_data_property("NBI_Prevailing Wind Direction", environmental_exposure_class, XSD.string)
create_data_property("NBI_Time of Wetness", environmental_exposure_class, XSD.decimal)
create_data_property("NBI_Total Precipitation", environmental_exposure_class, XSD.decimal)


try:
    g.serialize(destination="BridgeOntology1022.owl", format="pretty-xml", encoding="utf-8")
except Exception as e:
    print(f"An error occurred while saving the file: {e}")



# Newer version w/ neo4j compatability

In [2]:
!pip install neo4j rdflib-neo4j

  Obtaining dependency information for neo4j from https://files.pythonhosted.org/packages/75/4e/11813da186859070b0512e8071dac4796624ac4dc28e25e7c530df730d23/neo4j-6.0.2-py3-none-any.whl.metadata
  Obtaining dependency information for rdflib-neo4j from https://files.pythonhosted.org/packages/a0/a3/093d34de06174dcec25572b9256edcef076205086185e445748a9ed8a47b/rdflib_neo4j-1.1-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/325.8 kB ? eta -:--:--
   -------------------------------------- - 317.4/325.8 kB 9.9 MB/s eta 0:00:01
   ---------------------------------------- 325.8/325.8 kB 5.1 MB/s eta 0:00:00


In [11]:
import re
from rdflib import Graph, Literal, Namespace, RDF, RDFS
from rdflib.namespace import OWL, XSD
# Correct imports based on the documentation
from rdflib_neo4j import Neo4jStoreConfig, Neo4jStore, HANDLE_VOCAB_URI_STRATEGY
# NEW import for our native Cypher query
from neo4j import GraphDatabase

# --- 1. SET UP NEO4J CONNECTION DETAILS ---
# IMPORTANT: Change 'your_password' to your real Neo4j password
uri = "bolt://localhost:7687"
user = "neo4j"
password = "ILoveBridges"  # <--- !!! CHANGE THIS !!!
database = "neo4j"

# --- 2. NATIVE NEO4J CONNECTION TO CLEAR DB ---
print("Connecting directly to Neo4j to clear database...")
driver = None
try:
    # Use the native neo4j driver to run a Cypher query
    driver = GraphDatabase.driver(uri, auth=(user, password))
    with driver.session(database=database) as session:
        session.run("MATCH (n) DETACH DELETE n")
    print("Database cleared successfully.")
except Exception as e:
    print(f"\n--- NATIVE NEO4J ERROR ---")
    print(f"Could not clear database: {e}")
    print("Please check: Is Neo4j running? Is your password correct?")
    # We will exit if we can't clear the DB
    exit()
finally:
    if driver:
        driver.close()


# --- 3. RDFLIB-NEO4J CONNECTION TO LOAD SCHEMA ---
print(f"Connecting to Neo4j at {uri} with rdflib...")
try:
    # 1. Define connection details as a dictionary
    auth_data = {
        'uri': uri,
        'user': user,
        'pwd': password,
        'database': database
    }

    # 2. Define the Store Configuration
    config = Neo4jStoreConfig(
        auth_data=auth_data,
        handle_vocab_uri_strategy=HANDLE_VOCAB_URI_STRATEGY.IGNORE,
        batching=True # Use batching for efficiency
    )

    # 3. Initialize the Graph with the configured Neo4jStore
    g = Graph(store=Neo4jStore(config=config))
    
    # 4. We NO LONGER clear the DB here
    # print("Clearing Neo4j database...") # <-- Already done
    # g.update("MATCH (n) DETACH DELETE n") # <-- This was the broken line

    print("rdflib connection successful. Setting up namespaces...")
    
    # --- 4. SET UP GRAPH AND NAMESPACES ---
    BMO = Namespace("http://www.bridge-management-ontology.org/bmo#")
    g.bind("bmo", BMO)
    g.bind("owl", OWL)
    g.bind("rdfs", RDFS)
    g.bind("xsd", XSD)
    
    # --- 3. HELPER FUNCTION ---
    def create_data_property(column_name, domain_class, data_type=XSD.string):
        """
        Helper function to create a data property, clean its name,
        and link it to its domain class.
        """
        # Clean the name: remove special chars, use camelCase
        prop_name = re.sub(r'[^\w\s-]', '', column_name) # Remove special chars like %, ?, (
        prop_name_parts = re.split(r'[\s\(\).\-/]', prop_name)
        prop_name_camel = ''.join(x.capitalize() for x in prop_name_parts if x)
        
        if not prop_name_camel:
            return # Skip if name is invalid
    
        prop_name_camel = prop_name_camel[0].lower() + prop_name_camel[1:]
        
        prop_uri = BMO[prop_name_camel]
        g.add((prop_uri, RDF.type, OWL.DatatypeProperty))
        g.add((prop_uri, RDFS.label, Literal(column_name)))
        g.add((prop_uri, RDFS.domain, domain_class))
        g.add((prop_uri, RDFS.range, data_type))
    
    # --- 4. DEFINE CLASSES ---
    print("Defining classes and writing to Neo4j...")
    
    # Top-level classes
    asset_class = BMO.Asset
    g.add((asset_class, RDF.type, OWL.Class))
    g.add((asset_class, RDFS.label, Literal("Asset")))
    g.add((asset_class, RDFS.comment, Literal("A physical entity that is owned and has value.")))
    
    event_class = BMO.Event
    g.add((event_class, RDF.type, OWL.Class))
    g.add((event_class, RDFS.label, Literal("Event")))
    g.add((event_class, RDFS.comment, Literal("An occurrence that happens at a specific point in time.")))
    
    # Asset Layer
    bridge_class = BMO.Bridge
    g.add((bridge_class, RDF.type, OWL.Class))
    g.add((bridge_class, RDFS.subClassOf, asset_class))
    g.add((bridge_class, RDFS.label, Literal("Bridge")))
    
    bridge_element_class = BMO.BridgeElement
    g.add((bridge_element_class, RDF.type, OWL.Class))
    g.add((bridge_element_class, RDFS.subClassOf, asset_class))
    g.add((bridge_element_class, RDFS.label, Literal("Bridge Element")))
    
    # Environment & Exposure Layer
    location_class = BMO.Location
    g.add((location_class, RDF.type, OWL.Class))
    g.add((location_class, RDFS.label, Literal("Location")))
    g.add((location_class, RDFS.comment, Literal("A geographical area.")))
    
    weather_exposure_class = BMO.WeatherExposure
    g.add((weather_exposure_class, RDF.type, OWL.Class))
    g.add((weather_exposure_class, RDFS.label, Literal("Weather Exposure")))
    
    traffic_profile_class = BMO.TrafficProfile
    g.add((traffic_profile_class, RDF.type, OWL.Class))
    g.add((traffic_profile_class, RDFS.label, Literal("Traffic Profile")))
    
    # Deterioration & Maintenance Layer
    maintenance_activity_class = BMO.MaintenanceActivity
    g.add((maintenance_activity_class, RDF.type, OWL.Class))
    g.add((maintenance_activity_class, RDFS.subClassOf, event_class))
    g.add((maintenance_activity_class, RDFS.label, Literal("Maintenance Activity")))
    
    inspection_class = BMO.Inspection
    g.add((inspection_class, RDF.type, OWL.Class))
    g.add((inspection_class, RDFS.subClassOf, maintenance_activity_class))
    g.add((inspection_class, RDFS.label, Literal("Inspection")))
    
    maintenance_action_class = BMO.MaintenanceAction
    g.add((maintenance_action_class, RDF.type, OWL.Class))
    g.add((maintenance_action_class, RDFS.subClassOf, maintenance_activity_class))
    g.add((maintenance_action_class, RDFS.label, Literal("Maintenance Action")))
    
    deterioration_process_class = BMO.DeteriorationProcess
    g.add((deterioration_process_class, RDF.type, OWL.Class))
    g.add((deterioration_process_class, RDFS.label, Literal("Deterioration Process")))
    g.add((deterioration_process_class, RDFS.comment, Literal("A process (e.g., Corrosion, Scour) that degrades an asset.")))
    
    condition_rating_class = BMO.ConditionRating
    g.add((condition_rating_class, RDF.type, OWL.Class))
    g.add((condition_rating_class, RDFS.label, Literal("Condition Rating")))
    g.add((condition_rating_class, RDFS.comment, Literal("A rating assigned as a result of an inspection.")))
    
    deterioration_prediction_class = BMO.DeteriorationPrediction
    g.add((deterioration_prediction_class, RDF.type, OWL.Class))
    g.add((deterioration_prediction_class, RDFS.label, Literal("Deterioration Prediction")))
    
    # --- 5. DEFINE OBJECT PROPERTIES (Relationships) ---
    print("Defining object properties and writing to Neo4j...")
    
    # Asset Layer
    has_element_prop = BMO.hasElement
    is_part_of_prop = BMO.isPartOf
    g.add((has_element_prop, RDF.type, OWL.ObjectProperty))
    g.add((has_element_prop, RDFS.domain, bridge_class))
    g.add((has_element_prop, RDFS.range, bridge_element_class))
    g.add((is_part_of_prop, RDF.type, OWL.ObjectProperty))
    g.add((is_part_of_prop, RDFS.domain, bridge_element_class))
    g.add((is_part_of_prop, RDFS.range, bridge_class))
    g.add((has_element_prop, OWL.inverseOf, is_part_of_prop))
    g.add((is_part_of_prop, OWL.inverseOf, has_element_prop))
    
    # Environment Layer
    g.add((BMO.locatedIn, RDF.type, OWL.ObjectProperty))
    g.add((BMO.locatedIn, RDFS.domain, bridge_class))
    g.add((BMO.locatedIn, RDFS.range, location_class))
    g.add((BMO.hasExposure, RDF.type, OWL.ObjectProperty))
    g.add((BMO.hasExposure, RDFS.domain, location_class))
    g.add((BMO.hasExposure, RDFS.range, weather_exposure_class))
    g.add((BMO.hasTrafficProfile, RDF.type, OWL.ObjectProperty))
    g.add((BMO.hasTrafficProfile, RDFS.domain, bridge_class))
    g.add((BMO.hasTrafficProfile, RDFS.range, traffic_profile_class))
    g.add((BMO.trafficInducesStressOn, RDF.type, OWL.ObjectProperty))
    g.add((BMO.trafficInducesStressOn, RDFS.domain, traffic_profile_class))
    g.add((BMO.trafficInducesStressOn, RDFS.range, bridge_class))
    g.add((BMO.weatherInducesStressOn, RDF.type, OWL.ObjectProperty))
    g.add((BMO.weatherInducesStressOn, RDFS.domain, weather_exposure_class))
    g.add((BMO.weatherInducesStressOn, RDFS.range, bridge_class))
    
    # Deterioration & Maintenance Layer
    g.add((BMO.experiences, RDF.type, OWL.ObjectProperty))
    g.add((BMO.experiences, RDFS.domain, asset_class))
    g.add((BMO.experiences, RDFS.range, deterioration_process_class))
    g.add((BMO.observedBy, RDF.type, OWL.ObjectProperty))
    g.add((BMO.observedBy, RDFS.domain, deterioration_process_class))
    g.add((BMO.observedBy, RDFS.range, inspection_class))
    g.add((BMO.resultedIn, RDF.type, OWL.ObjectProperty))
    g.add((BMO.resultedIn, RDFS.domain, inspection_class))
    g.add((BMO.resultedIn, RDFS.range, condition_rating_class))
    g.add((BMO.triggers, RDF.type, OWL.ObjectProperty))
    g.add((BMO.triggers, RDFS.domain, condition_rating_class))
    g.add((BMO.triggers, RDFS.range, maintenance_action_class))
    g.add((BMO.isPerformedOn, RDF.type, OWL.ObjectProperty))
    g.add((BMO.isPerformedOn, RDFS.domain, maintenance_activity_class))
    g.add((BMO.isPerformedOn, RDFS.range, asset_class))
    
    # Prediction Layer
    g.add((BMO.predictionTriggersAction, RDF.type, OWL.ObjectProperty))
    g.add((BMO.predictionTriggersAction, RDFS.domain, deterioration_prediction_class))
    g.add((BMO.predictionTriggersAction, RDFS.range, maintenance_action_class))
    g.add((BMO.derivedFromWeather, RDF.type, OWL.ObjectProperty))
    g.add((BMO.derivedFromWeather, RDFS.domain, deterioration_prediction_class))
    g.add((BMO.derivedFromWeather, RDFS.range, weather_exposure_class))
    g.add((BMO.derivedFromTraffic, RDF.type, OWL.ObjectProperty))
    g.add((BMO.derivedFromTraffic, RDFS.domain, deterioration_prediction_class))
    g.add((BMO.derivedFromTraffic, RDFS.range, traffic_profile_class))
    g.add((BMO.predictsFor, RDF.type, OWL.ObjectProperty))
    g.add((BMO.predictsFor, RDFS.domain, deterioration_prediction_class))
    g.add((BMO.predictsFor, RDFS.range, asset_class))
    
    # --- 6. DEFINE DATA PROPERTIES ---
    print("Defining data properties and writing to Neo4j...")
    # Bridge
    create_data_property("STRUCTURE_NUMBER", bridge_class, XSD.string)
    create_data_property("LATEST_STRUCTURE_NUMBER", bridge_class, XSD.string)
    create_data_property("NBI_8 - Structure Number", bridge_class, XSD.string)
    create_data_property("NBI_2024 NBI Structure Number", bridge_class, XSD.string)
    create_data_property("NBI_27 - Year Built", bridge_class, XSD.gYear)
    create_data_property("NBI_106 - Year Reconstructed", bridge_class, XSD.gYear)
    create_data_property("NBI_Bridge Age (yr)", bridge_class, XSD.integer)
    create_data_property("NBI_43A - Main Span Material", bridge_class, XSD.string)
    create_data_property("NBI_43B - Main Span Design", bridge_class, XSD.string)
    create_data_property("NBI_44A - Approach Spans Material", bridge_class, XSD.string)
    create_data_property("NBI_44B - Approach Spans Design", bridge_class, XSD.string)
    create_data_property("NBI_45 - Number of Spans in Main Unit", bridge_class, XSD.integer)
    create_data_property("NBI_46 - Number of Approach Spans", bridge_class, XSD.integer)
    create_data_property("NBI_48 - Length of Maximum Span (ft.)", bridge_class, XSD.decimal)
    create_data_property("NBI_49 - Structure Length (ft.)", bridge_class, XSD.decimal)
    create_data_property("NBI_112 - NBIS Bridge Length", bridge_class, XSD.decimal)
    create_data_property("NBI_51 - Bridge Roadway Width Curb-to-Curb (ft.)", bridge_class, XSD.decimal)
    create_data_property("NBI_52 - Deck Width - Out to Out (ft.)", bridge_class, XSD.decimal)
    create_data_property("NBI_CAT29 - Deck Area (sq. ft.)", bridge_class, XSD.decimal)
    create_data_property("NBI_107 - Deck Structure Type Code", bridge_class, XSD.string)
    create_data_property("NBI_108A - Wearing Surface Type Code", bridge_class, XSD.string)
    create_data_property("NBI_108B - Membrane Type Code", bridge_class, XSD.string)
    create_data_property("NBI_108C - Deck Protection Code", bridge_class, XSD.string)
    create_data_property("NBI_31 - Design Load Descriptor", bridge_class, XSD.string)
    create_data_property("NBI_35 - Approach Roadway Width (ft.)", bridge_class, XSD.decimal)
    create_data_property("NBI_10 - Inventory Route Minimum Vertical Clearance (ft.)", bridge_class, XSD.decimal)
    create_data_property("OWNER_AGENCY", bridge_class, XSD.string)
    create_data_property("NBI_22 - Owner Agency", bridge_class, XSD.string)
    create_data_property("NBI_21 - Maintenance Responsibility", bridge_class, XSD.string)
    create_data_property("NBI_26 - Functional Classification of Inventory Route", bridge_class, XSD.string)
    create_data_property("NBI_104 - Highway System of Inventory Route", bridge_class, XSD.string)
    create_data_property("NBI_37 - Historical Significance Code", bridge_class, XSD.string)
    create_data_property("NBI_6A - Features Intersected", bridge_class, XSD.string)
    create_data_property("NBI_7 - Facility Carried By Structure", bridge_class, XSD.string)
    create_data_property("NBI_41 - Structure Operational Status Code", bridge_class, XSD.string)
    create_data_property("NBI_63 - Operating Rating Method Code", bridge_class, XSD.string)
    create_data_property("NBI_64 - Operating Rating (US tons)", bridge_class, XSD.decimal)
    create_data_property("NBI_65 - Inventory Rating Method Code", bridge_class, XSD.string)
    create_data_property("NBI_66 - Inventory Rating (US tons)", bridge_class, XSD.decimal)
    create_data_property("NBI_70 - Bridge Posting Code", bridge_class, XSD.string)
    create_data_property("NBI_113 - Scour Critical Bridge Value", bridge_class, XSD.string)
    
    # BridgeElement
    create_data_property("YEAR", bridge_element_class, XSD.gYear)
    create_data_property("ELEMENT_NUMBER", bridge_element_class, XSD.integer)
    create_data_property("ELEMENT_CATEGORY", bridge_element_class, XSD.string)
    create_data_property("PARENT_ELEMENT_NUMBER", bridge_element_class, XSD.integer)
    create_data_property("ELEMENT_NAME", bridge_element_class, XSD.string)
    create_data_property("ELEMENT_UNIT", bridge_element_class, XSD.string)
    create_data_property("TOTAL_QUANTITY", bridge_element_class, XSD.decimal)
    create_data_property("CONDITION_STATE_1", bridge_element_class, XSD.decimal)
    create_data_property("CS_PERCENT_1", bridge_element_class, XSD.decimal)
    create_data_property("CONDITION_STATE_2", bridge_element_class, XSD.decimal)
    create_data_property("CS_PERCENT_2", bridge_element_class, XSD.decimal)
    create_data_property("CONDITION_STATE_3", bridge_element_class, XSD.decimal)
    create_data_property("CS_PERCENT_3", bridge_element_class, XSD.decimal)
    create_data_property("CONDITION_STATE_4", bridge_element_class, XSD.decimal)
    create_data_property("CS_PERCENT_4", bridge_element_class, XSD.decimal)
    create_data_property("HI", bridge_element_class, XSD.decimal) # Health Index
    
    # Location
    create_data_property("NBI_1 - State Code", location_class, XSD.string)
    create_data_property("NBI_1 - State Name", location_class, XSD.string)
    create_data_property("NBI_3 - County Code", location_class, XSD.string)
    create_data_property("NBI_3 - County Name", location_class, XSD.string)
    create_data_property("NBI_4 - Place Code", location_class, XSD.string)
    create_data_property("NBI_City - InfoBridge Place Code", location_class, XSD.string)
    create_data_property("NBI_City - InfoBridge Place Name", location_class, XSD.string)
    create_data_property("NBI_9 - Location", location_class, XSD.string)
    create_data_property("NBI_16 - Latitude (decimal)", location_class, XSD.decimal)
    create_data_property("NBI_17 - Longitude (decimal)", location_class, XSD.decimal)
    create_data_property("ClimateZone", location_class, XSD.string)
    
    # WeatherExposure
    create_data_property("NBI_Average Relative Humidity", weather_exposure_class, XSD.decimal)
    create_data_property("NBI_Average Temperature", weather_exposure_class, XSD.decimal)
    create_data_property("NBI_Maximum Temperature", weather_exposure_class, XSD.decimal)
    create_data_property("NBI_Mean Wind Speed", weather_exposure_class, XSD.decimal)
    create_data_property("NBI_Minimum Temperature", weather_exposure_class, XSD.decimal)
    create_data_property("NBI_Number of Freeze-Thaw Cycles", weather_exposure_class, XSD.integer)
    create_data_property("NBI_Number of Snowfall Days", weather_exposure_class, XSD.integer)
    create_data_property("NBI_Number of Days with Measurable Precipitation", weather_exposure_class, XSD.integer)
    create_data_property("NBI_Number of Days with Temperature Below 0C", weather_exposure_class, XSD.integer)
    create_data_property("NBI_Prevailing Wind Direction", weather_exposure_class, XSD.string)
    create_data_property("NBI_Time of Wetness", weather_exposure_class, XSD.decimal)
    create_data_property("NBI_Total Precipitation", weather_exposure_class, XSD.decimal)
    
    # TrafficProfile
    create_data_property("NBI_29 - Average Daily Traffic", traffic_profile_class, XSD.integer)
    create_data_property("NBI_109 - Average Daily Truck Traffic (Percent ADT)", traffic_profile_class, XSD.decimal)
    create_data_property("NBI_30 - Year of Average Daily Traffic", traffic_profile_class, XSD.gYear)
    create_data_property("NBI_114 - Future Average Daily Traffic", traffic_profile_class, XSD.integer)
    create_data_property("NBI_115 - Year of Future Average Daily Traffic", traffic_profile_class, XSD.gYear)
    create_data_property("NBI_Computed - Average Daily Truck Traffic (Volume)", traffic_profile_class, XSD.integer)
    
    # Inspection
    create_data_property("NBI_Year", inspection_class, XSD.gYear)
    create_data_property("NBI_90 - Inspection Date", inspection_class, XSD.date)
    create_data_property("NBI_91 - Designated Inspection Frequency", inspection_class, XSD.integer)
    create_data_property("NBI_92A - Fracture Critical Details", inspection_class, XSD.string)
    create_data_property("NBI_92B - Underwater Inspection", inspection_class, XSD.string)
    create_data_property("NBI_92C - Other Special Inspection", inspection_class, XSD.string)
    create_data_property("NBI_93A - Fracture Critical Detail Date", inspection_class, XSD.date)
    create_data_property("NBI_93B - Underwater Inspection Date", inspection_class, XSD.date)
    create_data_property("NBI_93C - Other Special Inspection Date", inspection_class, XSD.date)
    
    # ConditionRating
    create_data_property("NBI_CAT10 - Bridge Condition", condition_rating_class, XSD.string)
    create_data_property("NBI_CAT23 - Condition Code", condition_rating_class, XSD.string)
    create_data_property("NBI_36A - Bridge Railings", condition_rating_class, XSD.string)
    create_data_property("NBI_36B - Transitions", condition_rating_class, XSD.string)
    create_data_property("NBI_36C - Approach Guardrail", condition_rating_class, XSD.string)
    create_data_property("NBI_36D - Bridge Guardrail Ends", condition_rating_class, XSD.string)
    create_data_property("NBI_58 - Deck Condition Rating", condition_rating_class, XSD.string)
    create_data_property("NBI_59 - Superstructure Condition Rating", condition_rating_class, XSD.string)
    create_data_property("NBI_60 - Substructure Condition Rating", condition_rating_class, XSD.string)
    create_data_property("NBI_61 - Channel and Channel Protection Condition Rating", condition_rating_class, XSD.string)
    create_data_property("NBI_62 - Culverts Condition Rating", condition_rating_class, XSD.string)
    create_data_property("NBI_67 - Structural Evaluation Appraisal", condition_rating_class, XSD.string)
    create_data_property("NBI_68 - Deck Geometry Appraisal", condition_rating_class, XSD.string)
    create_data_property("NBI_69 - Underclearance Appraisal Vertical and Horizontal", condition_rating_class, XSD.string)
    create_data_property("NBI_71 - Waterway Adequacy Appraisal", condition_rating_class, XSD.string)
    create_data_property("NBI_72 - Approach Alignment Appraisal", condition_rating_class, XSD.string)
    create_data_property("conditionValue", condition_rating_class, XSD.string)
    create_data_property("conditionScore", condition_rating_class, XSD.decimal)
    
    # MaintenanceAction
    create_data_property("NBI_94 - Bridge Improvement Cost", maintenance_action_class, XSD.decimal)
    create_data_property("NBI_76 - Length of Structure Improvement (ft.)", maintenance_action_class, XSD.decimal)
    create_data_property("NBI_95 - Roadway Improvement Cost", maintenance_action_class, XSD.decimal)
    create_data_property("NBI_96 - Total Project Cost", maintenance_action_class, XSD.decimal)
    create_data_property("NBI_97 - Year of Improvement Cost Estimate", maintenance_action_class, XSD.gYear)
    create_data_property("actionType", maintenance_action_class, XSD.string)
    create_data_property("actionDate", maintenance_action_class, XSD.date)
    create_data_property("actionCost", maintenance_action_class, XSD.decimal)
    
    # --- 8. CLOSE CONNECTION AND COMMIT ---
    # As per the docs, this is crucial when batching=True
    g.close(True)
    
    print("\n--- SUCCESS! ---")
    print("The ontology schema has been pushed and committed to your Neo4j database.")
    print("Open Neo4j Browser and run 'MATCH (n) RETURN n' to see the graph.")

except Exception as e:
    print("\n--- ERROR ---")
    print(f"An error occurred during the rdflib-neo4j process: {e}")
    print("\nPlease check:")
    print("1. Is your Neo4j database running?")
    print("2. Did you set the correct 'password' variable in this script?")
    print("3. Did you install the libraries? (!pip install neo4j rdflib-neo4j)")
    print(f"4. (If error persists): Check if your Neo4j URI is {uri}")

Connecting directly to Neo4j to clear database...
Database cleared successfully.
Connecting to Neo4j at bolt://localhost:7687 with rdflib...
Uniqueness constraint on :Resource(uri) found. 
                
                
The store is now: Open
rdflib connection successful. Setting up namespaces...
Defining classes and writing to Neo4j...
Defining object properties and writing to Neo4j...
Defining data properties and writing to Neo4j...
The store is now: Closed
IMPORTED 570 TRIPLES

--- SUCCESS! ---
The ontology schema has been pushed and committed to your Neo4j database.
Open Neo4j Browser and run 'MATCH (n) RETURN n' to see the graph.


# Data population

In [22]:
import pandas as pd
from neo4j import GraphDatabase
import re
import math

# --- 1. SET UP NEO4J CONNECTION DETAILS ---
# IMPORTANT: Change 'your_password' to your real Neo4j password
uri = "bolt://localhost:7687"
user = "neo4j"
password = "ILoveBridges"  # <--- !!! CHANGE THIS !!!
database = "neo4j"

# --- 2. SET UP CSV FILE PATH ---
# IMPORTANT: Change this to the FULL path of your CSV file
# Example (Windows): "C:/Users/YourUser/Documents/NBE_and_NBI_OK.csv"
# Example (Mac/Linux): "/home/youruser/documents/NBE_and_NBI_OK.csv"
csv_file_path = "oklahoma/NBE_and_NBI_OK.csv" # <--- !!! CHANGE THIS !!!

# --- 3. HELPER FUNCTION ---
def clean_data(data):
    """Cleans data for Neo4j import: converts NaT/NaN to None, strips strings."""
    if pd.isna(data):
        return None
    if isinstance(data, str):
        return data.strip()
    # Convert potential numpy types to standard python types
    if hasattr(data, 'item'):
        return data.item()
    return data

# --- 4. BATCH PROCESSING FUNCTION ---
def process_batch(session, batch_data):
    """
    Takes a list of data rows (as dicts) and creates the graph
    nodes and relationships using a Cypher UNWIND query.
    """
    
    # This is the main query. It takes a list of rows ($batch)
    # and iterates over it to create the full graph structure.
    # This is the main query. It takes a list of rows ($batch)
    # and iterates over it to create the full graph structure.
    cypher_query = """
    UNWIND $batch AS row
    
    // --- 1. Create/Merge the core :Bridge node ---
    // MERGE finds the node or creates it. ON CREATE sets properties only if it's new.
    MERGE (b:Bridge {structureNumber: row.structure_number})
    ON CREATE SET
        b.latestStructureNumber = row.latest_structure_number,
        b.yearBuilt = row.year_built,
        b.bridgeAge = row.bridge_age,
        b.mainSpanMaterial = row.main_span_material,
        b.mainSpanDesign = row.main_span_design,
        b.approachSpanMaterial = row.approach_span_material,
        b.approachSpanDesign = row.approach_span_design,
        b.numMainSpans = row.num_main_spans,
        b.numApproachSpans = row.num_approach_spans,
        b.maxSpanLength = row.max_span_length,
        b.structureLength = row.structure_length,
        b.nbisLength = row.nbis_length,
        b.roadwayWidth = row.roadway_width,
        b.deckWidth = row.deck_width,
        b.deckArea = row.deck_area,
        b.deckStructureType = row.deck_structure_type,
        b.wearingSurfaceType = row.wearing_surface_type,
        b.membraneType = row.membrane_type,
        b.deckProtection = row.deck_protection,
        b.designLoad = row.design_load,
        b.approachRoadwayWidth = row.approach_roadway_width,
        b.verticalClearance = row.vertical_clearance,
        b.ownerAgency = row.owner_agency,
        b.maintenanceResponsibility = row.maintenance_responsibility,
        b.functionalClass = row.functional_class,
        b.highwaySystem = row.highway_system,
        b.historicalSignificance = row.historical_significance,
        b.featuresIntersected = row.features_intersected,
        b.facilityCarried = row.facility_carried,
        b.operationalStatus = row.operational_status,
        b.operatingRatingMethod = row.operating_rating_method,
        b.operatingRating = row.operating_rating,
        b.inventoryRatingMethod = row.inventory_rating_method,
        b.inventoryRating = row.inventory_rating,
        b.postingCode = row.posting_code,
        b.scourCritical = row.scour_critical
    
    // --- 2. Create/Merge the :Location node and link it ---
    // We use a composite key for location to make it more unique
    WITH b, row
    WHERE row.state_name IS NOT NULL AND row.county_name IS NOT NULL
    MERGE (l:Location {name: row.county_name + ', ' + row.state_name})
    ON CREATE SET
        l.stateCode = row.state_code,
        l.stateName = row.state_name,
        l.countyCode = row.county_code,
        l.countyName = row.county_name,
        l.placeCode = row.place_code,
        l.infoBridgePlaceCode = row.info_bridge_place_code,
        l.infoBridgePlaceName = row.info_bridge_place_name,
        l.nbiLocation = row.nbi_location,
        l.latitude = row.latitude,
        l.longitude = row.longitude
    MERGE (b)-[:LOCATED_IN]->(l)

    // --- 3. Create/Merge the :WeatherExposure node and link it ---
    // We link weather to the Location, not the bridge, as it's a regional property
    WITH b, l, row
    WHERE row.state_name IS NOT NULL AND row.county_name IS NOT NULL // Re-check location exists
    MERGE (w:WeatherExposure {weatherID: row.county_name + ', ' + row.state_name})
    ON CREATE SET
        w.avgRelativeHumidity = row.avg_humidity,
        w.avgTemp = row.avg_temp,
        w.maxTemp = row.max_temp,
        w.meanWindSpeed = row.wind_speed,
        w.minTemp = row.min_temp,
        w.freezeThawCycles = row.freeze_thaw_cycles,
        w.snowfallDays = row.snowfall_days,
        w.precipitationDays = row.precipitation_days,
        w.daysBelowZeroC = row.days_below_zero,
        w.windDirection = row.wind_direction,
        w.timeOfWetness = row.time_of_wetness,
        w.totalPrecipitation = row.total_precipitation
    MERGE (l)-[:HAS_EXPOSURE]->(w)

    // --- 4. Create/Merge the :TrafficProfile node and link it ---
    // We create one traffic profile per bridge, per ADT year
    WITH b, l, w, row
    WHERE row.year_of_adt IS NOT NULL AND row.structure_number IS NOT NULL
    MERGE (t:TrafficProfile {trafficID: row.structure_number + '_' + row.year_of_adt})
    ON CREATE SET
        t.adt = row.adt,
        t.adttPercent = row.adtt_percent,
        t.yearOfAdt = row.year_of_adt,
        t.futureAdt = row.future_adt,
        t.yearOfFutureAdt = row.year_of_future_adt,
        t.adttVolume = row.adtt_volume
    MERGE (b)-[:HAS_TRAFFIC_PROFILE]->(t)

    // --- 5. Create/Merge the :Inspection node for this row ---
    // We MERGE this node to handle multiple rows (elements) per inspection
    WITH b, row
    WHERE row.inspection_date IS NOT NULL AND row.structure_number IS NOT NULL
    MERGE (i:Inspection:Event:MaintenanceActivity {inspectionID: row.structure_number + '_' + row.inspection_date})
    ON CREATE SET
        i.nbiYear = row.nbi_year,
        i.inspectionDate = row.inspection_date,
        i.inspFreq = row.insp_freq,
        i.fractureCritical = row.fracture_critical,
        i.underwaterInsp = row.underwater_insp,
        i.otherSpecialInsp = row.other_special_insp,
        i.fractureCriticalDate = row.fracture_critical_date,
        i.underwaterInspDate = row.underwater_insp_date,
        i.otherSpecialInspDate = row.other_special_insp_date
    MERGE (i)-[:IS_PERFORMED_ON]->(b) // Use the relationship from our ontology

    // --- 6. Create/Merge the :ConditionRating node from this inspection ---
    // We MERGE this node to handle multiple rows (elements) per inspection
    WITH b, i, row
    WHERE i.inspectionID IS NOT NULL // Ensure inspection was created
    // We create a unique condition rating *per inspection*
    MERGE (cr:ConditionRating {conditionRatingID: i.inspectionID})
    ON CREATE SET
        cr.bridgeCondition = row.bridge_condition,
        cr.conditionCode = row.condition_code,
        cr.bridgeRailings = row.bridge_railings,
        cr.transitions = row.transitions,
        cr.approachGuardrail = row.approach_guardrail,
        cr.approachGuardrailEnds = row.approach_guardrail_ends,
        cr.deckCondition = row.deck_condition,
        cr.superstructureCondition = row.superstructure_condition,
        cr.substructureCondition = row.substructure_condition,
        cr.channelCondition = row.channel_condition,
        cr.culvertCondition = row.culvert_condition,
        cr.structuralEval = row.structural_eval,
        cr.deckGeometry = row.deck_geometry,
        cr.underclearance = row.underclearance,
        cr.waterwayAdequacy = row.waterway_adequacy,
        cr.approachAlignment = row.approach_alignment
    MERGE (i)-[:RESULTED_IN]->(cr) // Link Inspection to its ConditionRating

    // --- 7. Create/Merge the :BridgeElement node for this row ---
    // We pass all variables forward
    WITH b, i, cr, row
    // We check all components of the elementID to prevent null errors
    WHERE row.element_name IS NOT NULL AND row.structure_number IS NOT NULL AND row.element_number IS NOT NULL AND row.year IS NOT NULL
    
    // We MERGE the element to handle duplicate rows
    MERGE (be:BridgeElement:Asset {elementID: row.structure_number + '_' + row.element_number + '_' + row.year})
    ON CREATE SET
        be.year = row.year,
        be.elementNumber = row.element_number,
        be.elementCategory = row.element_category,
        be.parentElementNumber = row.parent_element_number,
        be.elementName = row.element_name,
        be.unit = row.element_unit,
        be.totalQuantity = row.total_quantity,
        be.cs1 = row.cs1,
        be.cs1_percent = row.cs1_percent,
        be.cs2 = row.cs2,
        be.cs2_percent = row.cs2_percent,
        be.cs3 = row.cs3,
        be.cs3_percent = row.cs3_percent,
        be.cs4 = row.cs4,
        be.cs4_percent = row.cs4_percent,
        be.hi = row.hi
    
    // Create the relationships for the element
    MERGE (b)-[:HAS_ELEMENT]->(be) // Link Bridge to its Element
    MERGE (be)-[:INSPECTED_DURING]->(i) // Link Element to its Inspection

    // --- 8. Create/Merge :MaintenanceAction node and link it ---
    // *** FIX: We add a WITH clause here to pass the variables from Section 7 ***
    WITH b, i, be, cr, row
    // We check all components of the actionID to prevent null errors
    WHERE (row.improvement_cost IS NOT NULL OR row.total_project_cost IS NOT NULL)
        AND row.structure_number IS NOT NULL 
        AND row.year_of_cost IS NOT NULL
    
    MERGE (ma:MaintenanceAction:Event:MaintenanceActivity {actionID: row.structure_number + '_' + row.year_of_cost})
    ON CREATE SET
        ma.bridgeImprovementCost = row.improvement_cost,
        ma.improvementLength = row.improvement_length,
        ma.roadwayImprovementCost = row.roadway_cost,
        ma.totalProjectCost = row.total_project_cost,
        ma.yearOfCostEstimate = row.year_of_cost
    MERGE (ma)-[:IS_PERFORMED_ON]->(b)
    """
    
    # Run the query, passing the batch of data as a parameter
    session.run(cypher_query, batch=batch_data)

# --- 5. MAIN EXECUTION SCRIPT ---
def main():
    driver = None
    try:
        # --- Connect to Neo4j ---
        driver = GraphDatabase.driver(uri, auth=(user, password))
        with driver.session(database=database) as session:
            print("Successfully connected to Neo4j.")
            
            # --- Create Constraints (for speed and data integrity) ---
            print("Setting up database constraints (this is fast)...")
            
            constraints = [
                "CREATE CONSTRAINT IF NOT EXISTS FOR (b:Bridge) REQUIRE b.structureNumber IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (l:Location) REQUIRE l.name IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (w:WeatherExposure) REQUIRE w.weatherID IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (t:TrafficProfile) REQUIRE t.trafficID IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (i:Inspection) REQUIRE i.inspectionID IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (ma:MaintenanceAction) REQUIRE ma.actionID IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (be:BridgeElement) REQUIRE be.elementID IS UNIQUE",
                "CREATE CONSTRAINT IF NOT EXISTS FOR (cr:ConditionRating) REQUIRE cr.conditionRatingID IS UNIQUE"
            ]

            for constraint in constraints:
                session.run(constraint)
            print("Constraints are set.")

            # --- Load data from CSV using pandas ---
            print(f"Loading data from '{csv_file_path}'...")
            try:
                df = pd.read_csv(csv_file_path, low_memory=False)
            except FileNotFoundError:
                print(f"--- FATAL ERROR ---")
                print(f"File not found: {csv_file_path}")
                print("Please update the 'csv_file_path' variable in this script.")
                return
            
            print(f"Loaded {len(df)} rows from CSV.")
            
            # --- Prepare data for batching ---
            # Rename columns to match the Cypher query variables
            column_rename_map = {
                'YEAR': 'year', 'STRUCTURE_NUMBER': 'structure_number', 'LATEST_STRUCTURE_NUMBER': 'latest_structure_number',
                'OWNER_AGENCY': 'owner_agency', 'ELEMENT_CATEGORY': 'element_category',
                'PARENT_ELEMENT_NUMBER': 'parent_element_number', 'ELEMENT_NUMBER': 'element_number', 'ELEMENT_NAME': 'element_name',
                'ELEMENT_UNIT': 'element_unit', 'TOTAL_QUANTITY': 'total_quantity',
                'CONDITION_STATE_1': 'cs1', 'CS_PERCENT_1': 'cs1_percent',
                'CONDITION_STATE_2': 'cs2', 'CS_PERCENT_2': 'cs2_percent',
                'CONDITION_STATE_3': 'cs3', 'CS_PERCENT_3': 'cs3_percent',
                'CONDITION_STATE_4': 'cs4', 'CS_PERCENT_4': 'cs4_percent', 'HI': 'hi',
                'NBI_Year': 'nbi_year', 'NBI_1 - State Code': 'state_code', 'NBI_1 - State Name': 'state_name',
                'NBI_3 - County Code': 'county_code', 'NBI_3.csv - County Name': 'county_name',
                'NBI_4 - Place Code': 'place_code', 'NBI_City - InfoBridge Place Code': 'info_bridge_place_code',
                'NBI_City - InfoBridge Place Name': 'info_bridge_place_name', 'NBI_27 - Year Built': 'year_built',
                'NBI_29 - Average Daily Traffic': 'adt', 'NBI_43A - Main Span Material': 'main_span_material',
                'NBI_43B - Main Span Design': 'main_span_design', 'NBI_45 - Number of Spans in Main Unit': 'num_main_spans',
                'NBI_49 - Structure Length (ft.)': 'structure_length', 'NBI_6A - Features Intersected': 'features_intersected',
                'NBI_7 - Facility Carried By Structure': 'facility_carried', 'NBI_CAT10 - Bridge Condition': 'bridge_condition',
                'NBI_Bridge Age (yr)': 'bridge_age', 'NBI_CAT29 - Deck Area (sq. ft.)': 'deck_area',
                'NBI_9 - Location': 'nbi_location', 'NBI_16 - Latitude (decimal)': 'latitude',
                'NBI_17 - Longitude (decimal)': 'longitude', 'NBI_21 - Maintenance Responsibility': 'maintenance_responsibility',
                'NBI_37 - Historical Significance Code': 'historical_significance', 'NBI_106 - Year Reconstructed': 'year_reconstructed',
                'NBI_44A - Approach Spans Material': 'approach_span_material', 'NBI_44B - Approach Spans Design': 'approach_span_design',
                'NBI_46 - Number of Approach Spans': 'num_approach_spans', 'NBI_107 - Deck Structure Type Code': 'deck_structure_type',
                'NBI_108A - Wearing Surface Type Code': 'wearing_surface_type', 'NBI_108B - Membrane Type Code': 'membrane_type',
                'NBI_108C - Deck Protection Code': 'deck_protection', 'NBI_48 - Length of Maximum Span (ft.)': 'max_span_length',
                'NBI_52 - Deck Width - Out to Out (ft.)': 'deck_width', 'NBI_90 - Inspection Date': 'inspection_date',
                'NBI_91 - Designated Inspection Frequency': 'insp_freq', 'NBI_92A - Fracture Critical Details': 'fracture_critical',
                'NBI_92B - Underwater Inspection': 'underwater_insp', 'NBI_92C - Other Special Inspection': 'other_special_insp',
                'NBI_93A - Fracture Critical Detail Date': 'fracture_critical_date', 'NBI_93B - Underwater Inspection Date': 'underwater_insp_date',
                'NBI_93C - Other Special Inspection Date': 'other_special_insp_date', 'NBI_36A - Bridge Railings': 'bridge_railings',
                'NBI_36B - Transitions': 'transitions', 'NBI_36C - Approach Guardrail': 'approach_guardrail',
                'NBI_36D - Bridge Guardrail Ends': 'approach_guardrail_ends', 'NBI_58 - Deck Condition Rating': 'deck_condition',
                'NBI_59 - Superstructure Condition Rating': 'superstructure_condition', 'NBI_60 - Substructure Condition Rating': 'substructure_condition',
                'NBI_61 - Channel and Channel Protection Condition Rating': 'channel_condition', 'NBI_62 - Culverts Condition Rating': 'culvert_condition',
                'NBI_67 - Structural Evaluation Appraisal': 'structural_eval', 'NBI_68 - Deck Geometry Appraisal': 'deck_geometry',
                'NBI_69 - Underclearance Appraisal Vertical and Horizontal': 'underclearance', 'NBI_71 - Waterway Adequacy Appraisal': 'waterway_adequacy',
                'NBI_72 - Approach Alignment Appraisal': 'approach_alignment', 'NBI_113 - Scour Critical Bridge Value': 'scour_critical',
                'NBI_31 - Design Load Descriptor': 'design_load', 'NBI_41 - Structure Operational Status Code': 'operational_status',
                'NBI_63 - Operating Rating Method Code': 'operating_rating_method', 'NBI_64 - Operating Rating (US tons)': 'operating_rating',
                'NBI_65 - Inventory Rating Method Code': 'inventory_rating_method', 'NBI_66 - Inventory Rating (US tons)': 'inventory_rating',
                'NBI_7Examples - Bridge Posting Code': 'posting_code',
                'NBI_30 - Year of Average Daily Traffic': 'year_of_adt',
                'NBI_109 - Average Daily Truck Traffic (Percent ADT)': 'adtt_percent', 'NBI_114 - Future Average Daily Traffic': 'future_adt',
                'NBI_115 - Year of Future Average Daily Traffic': 'year_of_future_adt', 'NBI_94 - Bridge Improvement Cost': 'improvement_cost',
                'NBI_76 - Length of Structure Improvement (ft.)': 'improvement_length', 'NBI_95 - Roadway Improvement Cost': 'roadway_cost',
                'NBI_96 - Total Project Cost': 'total_project_cost', 'NBI_97 - Year of Improvement Cost Estimate': 'year_of_cost',
                'NBI_Computed - Average Daily Truck Traffic (Volume)': 'adtt_volume', 'NBI_CAT23 - Condition Code': 'condition_code',
                'NBI_Average Relative Humidity': 'avg_humidity', 'NBI_Average Temperature': 'avg_temp',
                'NBI_Maximum Temperature': 'max_temp', 'NBI_Mean Wind Speed': 'wind_speed',
                'NBI_Minimum Temperature': 'min_temp', 'NBI_Number of Freeze-Thaw Cycles': 'freeze_thaw_cycles',
                'NBI_Number of Snowfall Days': 'snowfall_days', 'NBI_Number of Days with Measurable Precipitation': 'precipitation_days',
                'NBI_Number of Days with Temperature Below 0C': 'days_below_zero',
                'NBI_Prevailing Wind Direction': 'wind_direction', 'NBI_Time of Wetness': 'time_of_wetness',
                'NBI_Total Precipitation': 'total_precipitation',
                'NBI_51 - Bridge Roadway Width Curb-to-Curb (ft.)': 'roadway_width',
                'NBI_10 - Inventory Route, Minimum Vertical Clearance (ft.)': 'vertical_clearance',
                'NBI_26 - Functional Classification of Inventory Route': 'functional_class',
                'NBI_104 - Highway System of Inventory Route': 'highway_system',
                'NBI_35 - Approach Roadway Width (ft.)': 'approach_roadway_width',
                'NBI_112 - NBIS Bridge Length': 'nbis_length',
                'NBI_3 - County Name': 'county_name', 
                'NBI_70 - Bridge Posting Code': 'posting_code'
            }
            
            # Keep only the columns we've mapped
            # Get the list of original column names we care about
            original_cols_to_keep = [col for col in column_rename_map.keys() if col in df.columns]
            
            # Select and rename in one step
            data_df = df[original_cols_to_keep].rename(columns=column_rename_map)

            # Convert all data to a list of dictionaries, cleaning as we go
            data_for_neo4j = [
                {key: clean_data(value) for key, value in row.items()}
                for row in data_df.to_dict('records')
            ]
            
            # --- Process in batches ---
            batch_size = 500
            total_rows = len(data_for_neo4j)
            
            print(f"Starting to import {total_rows} rows in batches of {batch_size}...")
            
            for i in range(0, total_rows, batch_size):
                batch = data_for_neo4j[i:min(i + batch_size, total_rows)]
                process_batch(session, batch)
                print(f"  ... processed rows {i + 1} to {min(i + batch_size, total_rows)}")
            
            print("\n--- SUCCESS! ---")
            print(f"Successfully imported {total_rows} data rows into Neo4j.")
            print("You can now run 'MATCH (n) RETURN n' in Neo4j Browser to see the full graph!")

    except Exception as e:
        print("\n--- FATAL ERROR ---")
        print(f"An error occurred: {e}")
        print("\nPlease check:")
        print("1. Is your Neo4j database running?")
        print("2. Did you set the correct 'password' variable in this script?")
        print("3. Is the 'csv_file_path' variable correct?")
        
    finally:
        if driver:
            driver.close()
            print("\nNeo4j connection closed.")

# --- RUN THE SCRIPT ---
if __name__ == "__main__":
    main()

Successfully connected to Neo4j.
Setting up database constraints (this is fast)...
Constraints are set.
Loading data from 'oklahoma/NBE_and_NBI_OK.csv'...
Loaded 101786 rows from CSV.
Starting to import 101786 rows in batches of 500...
  ... processed rows 1 to 500
  ... processed rows 501 to 1000
  ... processed rows 1001 to 1500
  ... processed rows 1501 to 2000
  ... processed rows 2001 to 2500
  ... processed rows 2501 to 3000
  ... processed rows 3001 to 3500
  ... processed rows 3501 to 4000
  ... processed rows 4001 to 4500
  ... processed rows 4501 to 5000
  ... processed rows 5001 to 5500
  ... processed rows 5501 to 6000
  ... processed rows 6001 to 6500
  ... processed rows 6501 to 7000
  ... processed rows 7001 to 7500
  ... processed rows 7501 to 8000
  ... processed rows 8001 to 8500
  ... processed rows 8501 to 9000
  ... processed rows 9001 to 9500
  ... processed rows 9501 to 10000
  ... processed rows 10001 to 10500
  ... processed rows 10501 to 11000
  ... processed 